In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_21.pickle

In [ ]:
%%PandasProfile
### cell 21 ###

def count_then_return_percent_33(df, col):
    total = df[col].count()
    return df[col].value_counts(dropna=False).mul(100 / total).round(1)

def combine_subset_of_data_from_multiple_years_33(question, x_axis_title, include_2017=None):
    # map each year to its responses DataFrame
    df_map = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_df_2019_cell10,
        '2018': responses_df_2018_cell10,
    }
    years = ['2018', '2019', '2020', '2021', '2022']
    if include_2017 is not None:
        df_map['2017'] = responses_df_2017
        years.insert(0, '2017')
    # build list of per-year DataFrames
    dfs = []
    for yr in years:
        s = count_then_return_percent_33(df_map[yr], question).sort_index()
        dfs.append(pd.DataFrame({'percentage': s, 'year': yr}))
    df_combined = pd.concat(dfs)
    # extract category index into a column
    df_combined[x_axis_title] = df_combined.index
    df_combined.columns = ['percentage', 'year', x_axis_title]
    return df_combined

question_of_interest_cell33 = 'For how many years have you been writing code and/or programming?'

# Clean and standardize 2018 responses
responses_df_2018_cell10.rename(
    columns={'How long have you been writing code to analyze data?': question_of_interest_cell33},
    inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33] = (
    responses_df_2018_cell10[question_of_interest_cell33].replace({
        '< 1 year': '< 1 years',
        'I have never written code but I want to learn': 'I have never written code',
        'I have never written code and I do not want to learn': 'I have never written code',
        '1-2 years': '1-3 years',
        '20-30 years': '20+ years',
        '30-40 years': '20+ years',
        '40+ years': '20+ years',
    })
)

# Clean and standardize 2019 responses
responses_df_2019_cell10.rename(
    columns={'How long have you been writing code to analyze data (at work or at school)?': question_of_interest_cell33},
    inplace=True
)
responses_df_2019_cell10[question_of_interest_cell33] = (
    responses_df_2019_cell10[question_of_interest_cell33].replace({'1-2 years': '1-3 years'})
)

# Clean and standardize 2020 responses
responses_df_2020[question_of_interest_cell33] = (
    responses_df_2020[question_of_interest_cell33].replace({'1-2 years': '1-3 years'})
)

title_for_x_axis_cell33 = ''
programming_df_combined = (
    combine_subset_of_data_from_multiple_years_33(
        question_of_interest_cell33, title_for_x_axis_cell33
    )
    .sort_values(by=['year', 'percentage'], ascending=True)
)
programming_df_combined.info()